# SimCLR on Tiny-ImageNet-200 (PyTorch)

Self-supervised contrastive learning (SimCLR, **alignment + uniformity** objective) on
**Tiny-ImageNet-200** (200 classes, 64×64 images). The encoder learns representations
with no labels; a frozen-feature **k-NN** classifier then measures how useful they are.

This is an upgraded version of the original CIFAR-10 notebook:

| | Baseline (CIFAR-10) | This notebook (Tiny-ImageNet-200) |
|---|---|---|
| Dataset | CIFAR-10 — 10 classes, 32² | Tiny-ImageNet-200 — 200 classes, 64² |
| Backbone | ResNet-18 (stock stem) | ResNet-18 (small-image stem: 3×3 conv1, one downsample) |
| Optimizer | SGD, lr = 0.06 | SGD + momentum 0.9 + weight-decay 1e-4 |
| LR schedule | fixed | cosine decay + linear warmup |
| Batch size | 256 | 512 |
| Precision | fp32 | AMP (mixed precision) |
| Epochs | 10 | 45 |
| k-NN eval | sklearn (k=20, cosine) | GPU k-NN (k=20, cosine) over 100k train feats |

The alignment/uniformity split (Wang & Isola) is kept as the contrastive objective:
alignment pulls positive pairs together, uniformity spreads features on the unit sphere.

In [ ]:
# Implementation
import os, time, math, random
import numpy as np
import torch
import torchvision
from torch import nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms, datasets
from PIL import Image

In [ ]:
# Reproducibility: seed Python, NumPy, and PyTorch RNGs.
# (As in the baseline, this does not guarantee bit-exact CUDA repeats: cuDNN
# autotuned convolutions and AMP introduce nondeterminism. cudnn.benchmark is
# left on for speed.)
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.benchmark = True

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# Configuration
DATA_ROOT = "datasets/tiny-imagenet-200"   # 200 classes, 64x64 (downloaded separately)
IMG       = 64
BATCH     = 512
EPOCHS    = 45
BASE_LR   = 0.5
WARMUP    = 5
WORKERS   = 8
KNN_K     = 20
# Tiny-ImageNet channel statistics
MEAN = [0.4802, 0.4481, 0.3975]
STD  = [0.2770, 0.2691, 0.2821]

In [ ]:
# Data augmentation: two independently-augmented views per image form a positive pair.
def rgb_loader(path):
    with open(path, "rb") as f:
        return Image.open(f).convert("RGB")  # some Tiny-ImageNet images are grayscale

simclr_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG, scale=(0.2, 1.0)),
    transforms.RandomHorizontalFlip(0.5),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
eval_tf = transforms.Compose([transforms.ToTensor(), transforms.Normalize(MEAN, STD)])

class TwoViews:
    def __init__(self, tf): self.tf = tf
    def __call__(self, x): return self.tf(x), self.tf(x)

In [ ]:
# Datasets. Train = 100k images over 200 classes (labels ignored for contrastive
# training). Val = 10k labelled images (labels read from val_annotations.txt), used
# only for k-NN evaluation.
train_contrastive = datasets.ImageFolder(
    f"{DATA_ROOT}/train", transform=TwoViews(simclr_tf), loader=rgb_loader)
class_to_idx = train_contrastive.class_to_idx

class TinyVal(Dataset):
    def __init__(self, root, class_to_idx, transform):
        self.transform = transform
        self.samples = []
        with open(f"{root}/val/val_annotations.txt") as f:
            for line in f:
                a = line.strip().split("\t")
                if len(a) >= 2 and a[1] in class_to_idx:
                    self.samples.append((f"{root}/val/images/{a[0]}", class_to_idx[a[1]]))
    def __len__(self): return len(self.samples)
    def __getitem__(self, i):
        path, y = self.samples[i]
        return self.transform(rgb_loader(path)), y

train_loader = DataLoader(train_contrastive, batch_size=BATCH, shuffle=True, drop_last=True,
                          num_workers=WORKERS, pin_memory=True, persistent_workers=True)
print(f"train: {len(train_contrastive)} imgs / {len(class_to_idx)} classes")

In [ ]:
# Model: ResNet-18 encoder with a small-image stem (3x3 conv1, a single downsample
# instead of the stock 7x7-stride-2 + maxpool, which throws away too much of a 64x64
# image) + a 2-layer MLP projection head into the contrastive space.
class SimCLR(nn.Module):
    def __init__(self, feature_dim=512, out_dim=128):
        super().__init__()
        net = torchvision.models.resnet18(weights=None)
        net.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        net.maxpool = nn.MaxPool2d(2)   # 64 -> 32
        net.fc = nn.Identity()          # keep global avg-pool -> 512-d feature
        self.backbone = net
        self.projector = nn.Sequential(
            nn.Linear(feature_dim, feature_dim),
            nn.ReLU(),
            nn.Linear(feature_dim, out_dim),
        )
    def forward(self, x):
        h = self.backbone(x)
        return self.projector(h)

model = SimCLR().to(device)
print("encoder params:", sum(p.numel() for p in model.backbone.parameters()))

In [ ]:
# Loss: alignment + uniformity (Wang & Isola), the baseline's contrastive objective.
class AlignUniformLoss(nn.Module):
    def __init__(self, alpha=2, t=2, lam=1):
        super().__init__()
        self.alpha, self.t, self.lam = alpha, t, lam
    def lalign(self, x, y):
        return (x - y).norm(dim=1).pow(self.alpha).mean()
    def lunif(self, x):
        return torch.pdist(x, p=2).pow(2).mul(-self.t).exp().mean().log()
    def forward(self, z_i, z_j):
        z_i = F.normalize(z_i, dim=1)
        z_j = F.normalize(z_j, dim=1)
        return self.lalign(z_i, z_j) + self.lam * (self.lunif(z_i) + self.lunif(z_j)) / 2

criterion = AlignUniformLoss()

In [ ]:
# Optimizer: SGD + momentum + weight decay, with a cosine LR schedule and linear warmup.
optimizer = torch.optim.SGD(model.parameters(), lr=BASE_LR, momentum=0.9, weight_decay=1e-4)
scaler = torch.cuda.amp.GradScaler()

def lr_at(epoch):
    if epoch < WARMUP:
        return BASE_LR * (epoch + 1) / WARMUP
    prog = (epoch - WARMUP) / max(1, EPOCHS - WARMUP)
    return 0.5 * BASE_LR * (1 + math.cos(math.pi * prog))

In [ ]:
# Training (mixed precision). Per-epoch loss is also appended to run_logs/train_live.log
# so a long run can be monitored while the notebook executes headless.
os.makedirs("run_logs", exist_ok=True)
print("Starting Training")
model.train()
for epoch in range(EPOCHS):
    lr = lr_at(epoch)
    for g in optimizer.param_groups:
        g["lr"] = lr
    total_loss, nb, t0 = 0.0, 0, time.time()
    for (x_i, x_j), _ in train_loader:
        x_i = x_i.to(device, non_blocking=True)
        x_j = x_j.to(device, non_blocking=True)
        with torch.cuda.amp.autocast():
            z_i = model(x_i)
            z_j = model(x_j)
        loss = criterion(z_i.float(), z_j.float())
        optimizer.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item(); nb += 1
    avg_loss = total_loss / nb
    line = f"epoch: {epoch:>02}, loss: {avg_loss:.5f}, lr: {lr:.4f}"
    print(line + f"  ({time.time()-t0:.1f}s)")
    with open("run_logs/train_live.log", "a") as lf:
        lf.write(line + "\n")

os.makedirs("checkpoints", exist_ok=True)
torch.save(model.state_dict(), "checkpoints/simclr_tiny.pth")

In [ ]:
# Evaluation: freeze the encoder, embed every train and val image, and run a cosine
# k-NN classifier (k=20) on the val set. Done on the GPU (100k x 10k similarities,
# chunked) so it takes seconds rather than minutes.
train_eval = datasets.ImageFolder(f"{DATA_ROOT}/train", transform=eval_tf, loader=rgb_loader)
val_eval = TinyVal(DATA_ROOT, class_to_idx, eval_tf)

@torch.no_grad()
def get_feats(dataset):
    loader = DataLoader(dataset, batch_size=512, num_workers=WORKERS, pin_memory=True)
    model.eval()
    feats, labels = [], []
    for x, y in loader:
        with torch.cuda.amp.autocast():
            f = model.backbone(x.to(device, non_blocking=True))
        feats.append(F.normalize(f.float(), dim=1))
        labels.append(y)
    return torch.cat(feats), torch.cat(labels).to(device)

Xtr, ytr = get_feats(train_eval)
Xte, yte = get_feats(val_eval)

correct = 0
for i in range(0, Xte.size(0), 1000):
    sims = Xte[i:i+1000] @ Xtr.T
    votes = ytr[sims.topk(KNN_K, dim=1).indices]
    pred = torch.mode(votes, dim=1).values
    correct += (pred == yte[i:i+1000]).sum().item()

acc = correct / Xte.size(0) * 100
print(f"KNN Accuracy: {acc:.2f}%")